# Unit 1 — The Graph Substrate: finding a city's arteries

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bgalon/geo-graph-learning/blob/main/unit-1-graph-substrate/geoai-graph-unit1.ipynb)

**Lesson context.** A road map is a *graph*: intersections are nodes, street segments are edges. Once a city is a graph, "which streets are the arteries?" stops being a hand-wave and becomes a measurable question. But the answer depends on two choices the analyst makes — *which centrality* defines "important," and *which graph* (intersections-as-nodes vs. streets-as-nodes) we measure on. This demo dramatizes the **artery question**: *looking only at the road map, which streets are the arteries — and how do we define "artery" rigorously enough for a computer to find them?*

**What you'll build.** For Jerusalem, you'll (1) load a real, self-extracted road graph; (2) compute three centralities (degree, closeness, betweenness) on the **primal** graph; (3) draw **one interactive map** (pan/zoom, switchable basemaps) with the three top-10% centralities as toggleable layers that *disagree*; (4) build the **dual** (named-street) graph live and overlay its betweenness against the primal's interactively. The punchline: *there is no neutral graph.* An **optional appendix** then makes that algorithmic — turn restrictions and fewest-turns routing.

**Prerequisites.** Basic graph vocabulary (node, edge, path, degree); comfort running a notebook. **No geospatial background assumed.**

**Data.** Road network derived from OpenStreetMap — © OpenStreetMap contributors, [ODbL](https://www.openstreetmap.org/copyright). This notebook is **self-contained**: it downloads a [Geofabrik](https://download.geofabrik.de/asia/israel-and-palestine.html) regional extract once, then cuts it to a Jerusalem bounding box inline with `pyrosm` — no pre-staged file, no separate prep script. The download is cached, so re-running the notebook is instant.


---

> **© 2026 Ben Galon. All rights reserved.** Part of the Geo-AI course (The Arena). Provided to enrolled students for course use; not for redistribution.
>
> **AI-assisted materials.** This notebook was drafted with AI assistance (Claude Code) using curated course context and reviewed by the instructor. It is a learning reference, not a guaranteed-correct key — verify before you rely on it.


## 1. Setup

On **Colab**, the cell below installs the Unit 1 stack (OSMnx, NetworkX, GeoPandas, Shapely, Matplotlib, requests, **pyrosm**, **igraph**, **folium**, **mapclassify**). On a **local** machine it is a no-op — you should already have run `uv sync --extra unit-1`. If install fails, see `KNOWN_ISSUES.md` in this unit folder.

In [ ]:
# Fetch the shared setup helper and install this unit's pinned requirements.
import os, urllib.request

if not os.path.exists("setup_colab.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/bgalon/geo-graph-learning/main/setup_colab.py",
        "setup_colab.py",
    )

from setup_colab import setup_unit
setup_unit("unit-1")

## 2. Smoke test — fail fast before any teaching content

Import every major library and run one trivial call each. This **asserts OSMnx v2.x** — all the code below uses the v2 module layout (`ox.simplification.*`, `ox.projection.*`, `ox.convert.*`) and will misbehave on the 2017-era v1 API. If anything here fails, fix the environment before continuing (see `KNOWN_ISSUES.md`).

In [ ]:
# Smoke test: import every major library + one trivial call each. Fail fast
# (cleanly) before any teaching content if the environment is not ready.
_smoke_error = None
try:
    import osmnx as ox
    import networkx as nx
    import geopandas as gpd
    import shapely
    import matplotlib
    import matplotlib.pyplot as plt
    import requests
    import pyrosm
    import igraph            # C-backed graph lib: fast exact centralities
    import folium            # interactive (Leaflet) maps
    import mapclassify       # choropleth binning for the folium overlays
    import gdown             # robust Google Drive download (data fallback)

    # Trivial calls.
    _ = nx.Graph()
    _ = gpd.GeoSeries.from_wkt(["POINT (0 0)"])
    _ = shapely.Point(0, 0)
    _ = pyrosm.get_data  # pyrosm importable (used for the inline cut below)
    _ = igraph.Graph()    # igraph importable (used for fast centralities)
    _ = gdown.download    # gdown importable (Drive fallback below)
    _ = folium.Map()      # folium importable (used for interactive maps)

    print("osmnx     ", ox.__version__)
    print("networkx  ", nx.__version__)
    print("geopandas ", gpd.__version__)
    print("shapely   ", shapely.__version__)
    print("pyrosm    ", pyrosm.__version__)
    print("igraph    ", igraph.__version__)
    print("folium    ", folium.__version__)

    # HARD GATE: this notebook uses the OSMnx v2 API.
    if int(ox.__version__.split(".")[0]) < 2:
        raise RuntimeError(
            f"This demo requires osmnx>=2.0 (got {ox.__version__}). "
            "See KNOWN_ISSUES.md: 'OSMnx v1 vs v2 API'."
        )
except Exception as e:
    _smoke_error = e

# Raise OUTSIDE the except block (no exception chaining) + `from None`, so a
# failure prints as ONE clean message instead of IPython's nested-traceback
# noise on Colab (py3.12 / IPython 9.x mis-formats a chained SystemExit).
if _smoke_error is not None:
    print("\n" + "=" * 70)
    print("SMOKE TEST FAILED:", _smoke_error)
    print("Do NOT continue — fix the environment first.")
    print("- If the setup cell above printed a WARN/404 for")
    print("  requirements/unit-1.txt, the unit deps never installed. Re-run the")
    print("  setup cell once that file is published (see KNOWN_ISSUES.md).")
    print("- If the error mentions pyrosm, see the 'pyrosm install' entry in")
    print("  unit-1-graph-substrate/KNOWN_ISSUES.md.")
    print("=" * 70)
    raise RuntimeError(f"Smoke test failed: {_smoke_error}") from None

print("\nSmoke test passed - environment is ready.")


## 3. Operations — construct a usable graph from OSM

**Concept (vocabulary):** *multigraph, graph simplification, projected vs. geographic CRS, connected components.*

Before we measure anything we need a clean graph. A raw OSM road network is a **non-planar, directed multigraph** with lots of interstitial nodes (curved streets), and its coordinates are in **degrees** — which silently breaks any length-based metric. The craft of "operations" is: simplify, consolidate, **project to meters**, and keep the largest connected component.

### 3a. How you'd pull a city live (SHOWCASE ONLY — not run)

This is the OSMnx v2 idiom for pulling a city straight from OpenStreetMap. We **show** it so you know how, but the rest of the notebook loads a pre-extracted cut so a live download never sits on the critical path. Leave `RUN_LIVE = False`.

In [ ]:
RUN_LIVE = False  # keep False so Run-All never hits the Overpass API

if RUN_LIVE:
    # The quick INTERACTIVE idiom: one Overpass query for one place.
    G_live = ox.graph_from_place("Jerusalem, Israel", network_type="drive")
    G_live = ox.simplification.simplify_graph(G_live)
    G_live = ox.projection.project_graph(G_live)
    print(G_live)

# Two extraction paths, and when to use each:
#   * Overpass via ox.graph_from_place (above) — the quick INTERACTIVE idiom
#     for ONE small place. Great in class, but it depends on a live API.
#   * Geofabrik regional .osm.pbf + pyrosm — the ROBUST BULK path: download a
#     regional extract once, then cut to a bbox at read time. No per-query API
#     dependency, reproducible, and what the NEXT cell actually runs to build
#     the Jerusalem graph this notebook analyzes.

### 3b. Build the Jerusalem primal graph inline (Geofabrik + pyrosm)

This is the **robust bulk** path. We download the Israel-and-Palestine
[Geofabrik](https://download.geofabrik.de/asia/israel-and-palestine.html)
extract **once** (cached on the runtime), then cut it to a Jerusalem bounding
box *at read time* with `pyrosm` — the country file is never held whole as a
graph. Then we run the OSMnx v2 craft on the result: **simplify → project to
UTM → consolidate intersections → keep the largest connected component**. No
pre-staged file, no separate prep script: the notebook makes its own data.

In [ ]:
# ── Inline data acquisition: download once, cut to Jerusalem, refine. ────────
# Self-contained: a fresh Colab runs this end-to-end. PRIMARY source is the
# Geofabrik regional extract (with retries + backoff). If Geofabrik is down
# (e.g. a 504 Gateway Timeout), we FALL BACK to a hosted Google Drive copy, and
# finally to an optional direct GraphML URL. First successful source wins.

# Jerusalem bounding box (WGS84 lon/lat): [min_lon, min_lat, max_lon, max_lat].
JERUSALEM_BBOX = [35.180, 31.740, 35.260, 31.810]

GEOFABRIK_PBF_URL = (
    "https://download.geofabrik.de/asia/israel-and-palestine-latest.osm.pbf"
)
PBF_PATH = "/content/israel-and-palestine.osm.pbf"  # cache target on Colab
if not os.path.isdir("/content"):
    PBF_PATH = "israel-and-palestine.osm.pbf"        # local fallback path

CONSOLIDATE_TOLERANCE_M = 10.0  # OSMnx v2 intersection-consolidation, meters

# Fallbacks (used ONLY if Geofabrik fails):
#  1. A hosted Google Drive copy. It may be the .pbf OR a prebuilt
#     jerusalem_primal.graphml — we sniff the downloaded file and handle either.
#  2. An optional direct raw-GraphML URL.
JERUSALEM_PRIMAL_DRIVE_ID = "18NSCmT1NwNVPl7uqvabuytKfAe3019yC"
JERUSALEM_PRIMAL_FALLBACK_URL = ""  # optional ".../jerusalem_primal.graphml"

import time
import requests


def _download_with_retries(url: str, dest: str, attempts: int = 4) -> str:
    """Download a large file with retries + exponential backoff.

    Handles transient Geofabrik 504s / timeouts. Writes atomically (.part →
    dest) so a failed attempt never leaves a half-file behind. Cached on re-run.
    """
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"using cached file: {dest} ({os.path.getsize(dest)/1e6:.0f} MB)")
        return dest
    last = None
    for i in range(1, attempts + 1):
        try:
            print(f"downloading (attempt {i}/{attempts}): {url}")
            with requests.get(url, stream=True, timeout=600) as r:
                r.raise_for_status()
                tmp = dest + ".part"
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
                os.replace(tmp, dest)
            print(f"saved {dest} ({os.path.getsize(dest)/1e6:.0f} MB)")
            return dest
        except Exception as exc:  # noqa: BLE001
            last = exc
            wait = 2 ** i
            print(f"  attempt {i} failed ({exc}); retrying in {wait}s…")
            time.sleep(wait)
    raise RuntimeError(f"download failed after {attempts} attempts: {last}")


def _refine_pbf_to_primal(pbf_path: str) -> "nx.MultiDiGraph":
    """pyrosm bbox cut → OSMnx v2 simplify → project (UTM) → consolidate → LCC.

    This is the Section 3 'operations' teaching content, shared by the Geofabrik
    path and the Drive-.pbf fallback path.
    """
    from pyrosm import OSM

    print(f"cutting to Jerusalem bbox {JERUSALEM_BBOX} with pyrosm…")
    osm = OSM(pbf_path, bounding_box=JERUSALEM_BBOX)
    nodes, edges = osm.get_network(network_type="driving", nodes=True)
    if nodes is None or edges is None or len(edges) == 0:
        raise RuntimeError("pyrosm returned no driving network for the bbox.")
    G = osm.to_graph(nodes, edges, graph_type="networkx", osmnx_compatible=True)
    if "crs" not in G.graph:
        G.graph["crs"] = "epsg:4326"
    print(f"MultiDiGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    try:
        G = ox.simplification.simplify_graph(G)
    except Exception as exc:  # already-simple graphs raise; that's fine
        print(f"simplify skipped ({exc})")
    G = ox.projection.project_graph(G)  # to UTM (meters) BEFORE consolidating
    try:
        G = ox.simplification.consolidate_intersections(
            G, tolerance=CONSOLIDATE_TOLERANCE_M, rebuild_graph=True,
            dead_ends=False,
        )
    except Exception as exc:
        print(f"consolidate skipped ({exc})")

    largest = max(nx.weakly_connected_components(G), key=len)
    G = G.subgraph(largest).copy()
    print(f"LCC: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G


def _build_primal_from_geofabrik() -> "nx.MultiDiGraph":
    _download_with_retries(GEOFABRIK_PBF_URL, PBF_PATH)
    return _refine_pbf_to_primal(PBF_PATH)


def _build_primal_from_drive(file_id: str) -> "nx.MultiDiGraph":
    """Fallback: pull a hosted copy from Google Drive (GraphML or .pbf).

    gdown handles public-file download incl. the large-file confirm token. We
    sniff the first bytes: XML → a prebuilt GraphML (load directly); otherwise
    treat it as an OSM .pbf and run the same pyrosm refine pipeline.
    """
    import gdown

    dest = "jerusalem_fallback.bin"
    print(f"FALLBACK: downloading hosted copy from Google Drive (id={file_id})…")
    path = gdown.download(id=file_id, output=dest, quiet=False)
    if not path or not os.path.exists(path):
        raise RuntimeError("gdown could not download the Drive fallback file.")
    with open(path, "rb") as f:
        head = f.read(64).lstrip()
    if head.startswith(b"<"):  # XML → GraphML
        print("Drive file looks like GraphML → loading with OSMnx")
        return ox.io.load_graphml(path)
    print("Drive file looks like an OSM .pbf → cutting with pyrosm")
    return _refine_pbf_to_primal(path)


def _build_primal_from_url(url: str) -> "nx.MultiDiGraph":
    dest = "jerusalem_primal.graphml"
    if not os.path.exists(dest):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(dest, "wb") as f:
            f.write(r.content)
    return ox.io.load_graphml(dest)


# Try each source in order; first success wins.
_sources = [("Geofabrik (primary)", _build_primal_from_geofabrik)]
if JERUSALEM_PRIMAL_DRIVE_ID:
    _sources.append(
        ("Google Drive fallback",
         lambda: _build_primal_from_drive(JERUSALEM_PRIMAL_DRIVE_ID)))
if JERUSALEM_PRIMAL_FALLBACK_URL:
    _sources.append(
        ("GraphML URL fallback",
         lambda: _build_primal_from_url(JERUSALEM_PRIMAL_FALLBACK_URL)))

G = None
_errors = []
for _name, _fn in _sources:
    try:
        print(f"--- source: {_name} ---")
        G = _fn()
        print(f"✓ built Jerusalem primal from: {_name}")
        break
    except Exception as exc:  # noqa: BLE001
        print(f"✗ {_name} failed: {exc}")
        _errors.append(f"{_name}: {exc}")

# Clean failure OUTSIDE the loop (no chained SystemExit → no IPython noise).
if G is None:
    print("\n" + "=" * 70)
    print("Could not build the Jerusalem graph from any source:")
    for _e in _errors:
        print("  -", _e)
    print("See unit-1-graph-substrate/KNOWN_ISSUES.md "
          "(Geofabrik-download / pyrosm-install entries).")
    print("=" * 70)
    raise RuntimeError("Jerusalem graph build failed (all sources exhausted).") from None

print()
print(G)


### 3c. Inspect the operations craft

Three adjectives on the live object — **directed, multigraph, non-planar** — plus a CRS sanity check. The single most common tutorial bug is computing centrality on an **unprojected** graph: edge "lengths" are then in degrees and every distance-weighted metric is silently wrong.

In [ ]:
print("type        :", type(G).__name__, "(directed multigraph)")
print("nodes       :", G.number_of_nodes())
print("edges       :", G.number_of_edges())
print("CRS         :", G.graph.get("crs"))

# CRS gotcha check: a projected CRS measures in meters, not degrees.
crs = str(G.graph.get("crs", "")).lower()
is_projected = ("utm" in crs) or ("epsg:326" in crs) or ("epsg:327" in crs)
print("projected?  :", is_projected, "(must be True for length-weighted metrics)")

# Largest connected component — centrality is computed on the LCC.
n_components = nx.number_weakly_connected_components(G)
print("components  :", n_components, "(we measure on the largest)")

In [ ]:
# Quick look at the network.
fig, ax = ox.plot.plot_graph(
    G, node_size=0, edge_color="#3b6ea5", edge_linewidth=0.6,
    bgcolor="white", show=False, close=False,
)
ax.set_title("Jerusalem drive network (primal graph)")
plt.show()

## 4. Operationalizing importance — three centralities on the primal

**Concept (vocabulary):** *degree, closeness, betweenness centrality on the primal LCC.*

There is a **family of five** centrality measures for street networks, each answering a different plain-language question:

| Centrality | The question it answers |
|---|---|
| **Degree** | How many streets meet here? (local connectivity) |
| **Closeness** | How near is this to everywhere else? (accessibility) |
| **Betweenness** | How often does the shortest path *pass through* here? (through-movement) |
| Straightness | How close to a straight line are routes through here? |
| Information | How much does removing this damage efficiency? |

We **teach five, compute three** — degree, closeness, betweenness — on the LCC, length-weighted. Straightness and information are named and left to the reading (and to "where to go next"). **Betweenness is our first hypothesis for "artery"** (Freeman 1977).

In [ ]:
# Work on the largest connected component as an undirected, simple, length-
# weighted Graph (collapses the multigraph for clean centrality).
largest = max(nx.weakly_connected_components(G), key=len)
G_lcc = G.subgraph(largest).copy()

GG = nx.Graph()
for u, v, data in G_lcc.edges(data=True):
    w = float(data.get("length", 1.0))
    if (not GG.has_edge(u, v)) or w < GG[u][v]["length"]:
        GG.add_edge(u, v, length=w)
print(f"LCC simple graph: {GG.number_of_nodes()} nodes, {GG.number_of_edges()} edges")

In [ ]:
# Degree — local connectivity (cheap). NetworkX is instant here, so we keep it.
deg = dict(GG.degree())

# Closeness + betweenness via igraph (C-backed) instead of pure-Python NetworkX.
# Teaching aside: NetworkX is clear but O(V*E); igraph (C-backed) gives the SAME
# EXACT values in seconds — the right tool when the graph grows. On a few-thousand
# -node LCC, nx.closeness_centrality + nx.betweenness_centrality can take ~20 min;
# igraph returns identical numbers in seconds.
import igraph as ig

# Convert the simple, length-weighted graph GG to igraph ONCE, preserving a
# node-index <-> node-id mapping so we can map results back onto the original ids
# (the rest of the notebook — top-decile masks, maps — is then unchanged).
node_ids = list(GG.nodes())                         # igraph index i  ->  node_ids[i]
idx_of = {n: i for i, n in enumerate(node_ids)}     # node id         ->  igraph index
edges_ig = [(idx_of[u], idx_of[v]) for u, v in GG.edges()]
weights = [float(GG[u][v]["length"]) for u, v in GG.edges()]
g = ig.Graph(n=len(node_ids), edges=edges_ig, directed=False)
g.es["length"] = weights

# Closeness: igraph's default normalized=True gives (n-1)/sum(dist) on the LCC,
# which matches nx.closeness_centrality on a single connected component.
clo_vals = g.closeness(weights="length", normalized=True)
clo = {node_ids[i]: clo_vals[i] for i in range(len(node_ids))}

# Betweenness: igraph returns RAW (unnormalized) shortest-path counts. The demo
# uses NetworkX-style NORMALIZED betweenness, so rescale by 2/((n-1)(n-2)) — the
# exact undirected normalization nx.betweenness_centrality applies — so values
# match nx.betweenness_centrality(GG, weight="length", normalized=True).
btw_raw = g.betweenness(weights="length", directed=False)
n = g.vcount()
scale = (2.0 / ((n - 1) * (n - 2))) if n > 2 else 1.0
btw = {node_ids[i]: btw_raw[i] * scale for i in range(len(node_ids))}

print("computed degree (networkx), closeness + betweenness (igraph, C-backed)")
print(f"  betweenness normalization: raw * 2/((n-1)(n-2)) with n={n}")

### 4a. Degree up close — distribution + a degree-coloured map

Before comparing the three centralities, look at the simplest one on its own.
**Degree** = how many streets meet at an intersection. Two views: the **degree
distribution** (a histogram — almost all intersections are 3- or 4-way, so the
primal road graph is *near-regular*, unlike the scale-free dual we build in
Section 6), and an **interactive map coloured by degree** (pan/zoom; toggle
OpenStreetMap / Esri satellite / blank-topology) so you can see *where* the
high-degree junctions sit.

In [ ]:
# Degree distribution — most intersections are 3- or 4-way (near-regular primal).
import matplotlib.pyplot as plt
from collections import Counter

deg_counts = Counter(deg.values())
ks = sorted(deg_counts)
mean_deg = sum(deg.values()) / len(deg)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(ks, [deg_counts[k] for k in ks], color="#1f78b4")
axes[0].set(xlabel="degree", ylabel="# intersections", title="Degree distribution (counts)")
axes[1].bar(ks, [deg_counts[k] / len(deg) for k in ks], color="#1f78b4")
axes[1].set(xlabel="degree", ylabel="fraction of intersections",
            title="Degree distribution (normalised)")
fig.suptitle(f"Primal road graph is near-regular — mean degree {mean_deg:.2f}; "
             f"most nodes are 3- or 4-way", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Interactive map coloured by degree (continuous), pan/zoom + basemap toggle.
# CRS gotcha: degree is computed on the projected graph; folium needs WGS84, so
# we reproject the node geometry to EPSG:4326 FOR RENDERING ONLY.
import folium
import branca.colormap as bcm

deg_nodes = ox.convert.graph_to_gdfs(G_lcc, edges=False).to_crs(4326)
deg_nodes["degree"] = [int(deg.get(nid, 0)) for nid in deg_nodes.index]
deg_edges = ox.convert.graph_to_gdfs(G_lcc, nodes=False).to_crs(4326)

dmin, dmax = int(deg_nodes["degree"].min()), int(deg_nodes["degree"].max())
cmap = bcm.linear.YlOrRd_09.scale(dmin, dmax)
cmap.caption = "Node degree (number of streets meeting at the intersection)"

_ESRI = ("https://server.arcgisonline.com/ArcGIS/rest/services/"
         "World_Imagery/MapServer/tile/{z}/{y}/{x}")
_BLANK = ("data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwC"
          "AAAAC0lEQVR42mNk+A8AAQUBAScY42YAAAAASUVORK5CYII=")

center = [deg_nodes.geometry.y.mean(), deg_nodes.geometry.x.mean()]
dm = folium.Map(location=center, zoom_start=14, tiles=None, control_scale=True)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(dm)
folium.TileLayer(tiles=_ESRI, attr="Tiles © Esri", name="Esri satellite",
                 overlay=False, control=True).add_to(dm)
folium.TileLayer(tiles=_BLANK, attr="blank", name="Blank (topology only)",
                 overlay=False, control=True).add_to(dm)

# Faint full network so the degree dots sit on the street geometry.
folium.GeoJson(
    deg_edges[["geometry"]],
    style_function=lambda _f: {"color": "#cccccc", "weight": 1, "opacity": 0.6},
    name="Full network (faint)",
).add_to(dm)

# One dot per intersection, coloured + sized by degree.
for _, row in deg_nodes.iterrows():
    d = row["degree"]
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2 + (d - dmin),               # higher degree = bigger dot
        color=None, weight=0, fill=True,
        fill_color=cmap(d), fill_opacity=0.9,
        tooltip=f"degree {d}",
    ).add_to(dm)

cmap.add_to(dm)                              # colour-bar legend
folium.LayerControl(collapsed=False).add_to(dm)
dm


## 5. Selection — one interactive map, three toggleable centralities

**Concept (vocabulary):** *select and visualize the top-10% of nodes under each definition — matching metric to question.*

Same city, same graph, three definitions of "important." Instead of three static panels we draw **one interactive folium (Leaflet) map** you can pan and zoom, with the three centralities as **toggleable overlay layers** (top-10% highlighted) so you can flip between definitions live. Three basemaps are switchable too: **OpenStreetMap**, **Esri satellite imagery**, and a **blank background** (pure topology, no map underneath).

Watch the layers **disagree**: degree lights up dense junction clusters, closeness lights up the geographic core, betweenness lights up the through-routes. The betweenness layer is the one that *looks like* arteries — hold that thought for Section 7.

**CRS gotcha (critical).** The metrics were computed on the **UTM-projected** graph (meters) — that is correct and we do **not** recompute them. But folium/Leaflet needs **WGS84 lat/lon**, so below we build GeoDataFrames of the nodes and edges, attach the centrality columns, and **reproject to EPSG:4326 only for rendering**.

In [ ]:
import numpy as np
import folium

# --- 1. Top-decile masks per centrality (computed on the projected graph). ---
def top_decile_mask(scores: dict, nodes) -> dict:
    vals = np.array([scores.get(n, 0.0) for n in nodes])
    cutoff = np.quantile(vals, 0.90)
    return {n: (scores.get(n, 0.0) >= cutoff) for n in nodes}

CENTRALITIES = {"Degree": deg, "Closeness": clo, "Betweenness": btw}
nodes = list(G_lcc.nodes())
top_masks = {label: top_decile_mask(s, nodes) for label, s in CENTRALITIES.items()}

# --- 2. CRS gotcha: metrics are on the UTM graph; folium needs WGS84. ---------
# Build node + edge GeoDataFrames, attach centrality columns, reproject to 4326
# FOR RENDERING ONLY. We do NOT recompute any metric on the unprojected graph.
nodes_gdf, edges_gdf = ox.convert.graph_to_gdfs(G_lcc)
nodes_gdf = nodes_gdf.to_crs(4326)            # reproject for Leaflet
edges_gdf = edges_gdf.to_crs(4326)
for label, scores in CENTRALITIES.items():
    nodes_gdf[label] = [scores.get(n, 0.0) for n in nodes_gdf.index]
    nodes_gdf[f"{label}_top"] = [top_masks[label][n] for n in nodes_gdf.index]

center = [nodes_gdf.geometry.y.mean(), nodes_gdf.geometry.x.mean()]

# --- 3. One map; three basemaps; three toggleable centrality overlays. -------
m = folium.Map(location=center, zoom_start=14, tiles=None, control_scale=True)

# Basemaps (radio-button group: exactly one shows at a time).
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/"
          "World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri", name="Esri satellite", overlay=False, control=True,
).add_to(m)
# Blank background = pure topology with no map underneath (1x1 transparent png).
_BLANK = ("data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwC"
          "AAAAC0lEQVR42mNk+A8AAQUBAScY42YAAAAASUVORK5CYII=")
folium.TileLayer(tiles=_BLANK, attr="blank", name="Blank (topology only)",
                 overlay=False, control=True).add_to(m)

# Faint full network once, as a non-toggleable backdrop so topology is visible
# on every layer (kept light: a single GeoJson of the edge lines).
folium.GeoJson(
    edges_gdf[["geometry"]],
    style_function=lambda _f: {"color": "#bbbbbb", "weight": 1, "opacity": 0.6},
    name="Full network (faint)", control=True,
).add_to(m)

# One toggleable overlay per centrality: full net faint + top-10% bold markers.
COLORS = {"Degree": "#1f78b4", "Closeness": "#33a02c", "Betweenness": "#e31a1c"}
for label in CENTRALITIES:
    fg = folium.FeatureGroup(name=f"Top 10% — {label}",
                             show=(label == "Betweenness"))  # default-on layer
    top = nodes_gdf[nodes_gdf[f"{label}_top"]]
    for _, row in top.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=4, color=COLORS[label], fill=True,
            fill_color=COLORS[label], fill_opacity=0.85, weight=0,
            tooltip=f"{label}: {row[label]:.4g}",
        ).add_to(fg)
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

## 6. Modeling — build the dual live, overlay primal vs. dual betweenness

**Concept (vocabulary):** *build the dual (named-street) graph; compute betweenness on it; overlay against primal betweenness — there is no neutral graph.*

**The naming trap, stated plainly.** The quick `nx.line_graph(G)` that many tutorials call "the dual" is **NOT** the Porta/Crucitti/Latora dual. The real dual uses **Intersection Continuity Negotiation**: a *named street*, spanning many segments, is a single node; two streets are linked when they share an intersection. So we **build it ourselves**, live.

In [ ]:
def build_named_street_dual(G):
    """Named-street / Intersection-Continuity-Negotiation (ICN) dual.

    NOT nx.line_graph(G). Three steps:
      1. group primal edges by street `name` (unnamed segments each become
         their own street so nothing is dropped);
      2. collapse each named street to ONE street-node;
      3. link two street-nodes when their member edges share a primal node.
    Returns (dual_graph, edge_to_street) so we can map results back.
    """
    edge_to_street = {}
    street_members = {}
    anon = 0
    for u, v, k, data in G.edges(keys=True, data=True):
        name = data.get("name")
        if isinstance(name, list):
            name = name[0] if name else None
        if not name:
            anon += 1
            street = f"__unnamed_{anon}"
        else:
            street = str(name)
        edge_to_street[(u, v, k)] = street
        street_members.setdefault(street, set()).update((u, v))

    D = nx.Graph()
    D.add_nodes_from(street_members.keys())

    node_to_streets = {}
    for street, member_nodes in street_members.items():
        for n in member_nodes:
            node_to_streets.setdefault(n, set()).add(street)
    for n, streets in node_to_streets.items():
        streets = list(streets)
        for i in range(len(streets)):
            for j in range(i + 1, len(streets)):
                D.add_edge(streets[i], streets[j])
    return D, edge_to_street

D, edge_to_street = build_named_street_dual(G_lcc)
print(f"dual: {D.number_of_nodes()} street-nodes, {D.number_of_edges()} links")

In [ ]:
# ── The naming trap, made concrete: nx.line_graph(G) is NOT the ICN dual ──────
# The markdown above warned that the quick `nx.line_graph(G)` many tutorials
# call "the dual" is NOT the Porta/Crucitti/Latora named-street dual. Show it
# two ways: (1) count nodes on the REAL graph, (2) draw both on a toy junction.
import matplotlib.pyplot as plt

L_naive = nx.line_graph(G_lcc)   # naive "dual": ONE node per directed segment
print(f"{'representation':<26}{'nodes':>8}{'edges':>9}")
print(f"{'primal (intersections)':<26}{G_lcc.number_of_nodes():>8}{G_lcc.number_of_edges():>9}")
print(f"{'line_graph (segments)':<26}{L_naive.number_of_nodes():>8}{L_naive.number_of_edges():>9}"
      "   <- one node per SEGMENT")
print(f"{'true ICN dual (streets)':<26}{D.number_of_nodes():>8}{D.number_of_edges():>9}"
      "   <- one node per NAMED STREET")
ratio = L_naive.number_of_nodes() / max(D.number_of_nodes(), 1)
print(f"\nThe line graph has ~{ratio:.0f}x more nodes than the true dual: it never applies\n"
      "Intersection Continuity Negotiation, so every little segment stays its own\n"
      "node instead of collapsing into the named street it belongs to.")

# ── Toy junction: 'Main St' (A-B-C-D, 3 segments) crossed by '1st Ave' (E-B-F).
P_toy = nx.Graph()
P_toy.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("E", "B"), ("B", "F")])
seg_street = {("A", "B"): "Main St", ("B", "C"): "Main St", ("C", "D"): "Main St",
              ("E", "B"): "1st Ave", ("B", "F"): "1st Ave"}
street_of = lambda e: seg_street.get(e) or seg_street.get((e[1], e[0]))
COL = {"Main St": "#1f5fae", "1st Ave": "#d62728"}

P_pos = {"A": (-2, 0), "B": (0, 0), "C": (2, 0), "D": (4, 0), "E": (0, 2), "F": (0, -2)}
L_toy = nx.line_graph(P_toy)                        # 5 segment-nodes
L_pos0 = {("A", "B"): (-2.2, 0.2), ("B", "C"): (0.6, 0.6), ("C", "D"): (2.6, 0.2),
          ("E", "B"): (-0.4, 1.7), ("B", "F"): (-0.4, -1.7)}
L_pos = {n: (L_pos0.get(n) or L_pos0.get((n[1], n[0]))) for n in L_toy.nodes()}
D_toy = nx.Graph(); D_toy.add_edge("Main St", "1st Ave")   # 2 street-nodes
D_pos = {"Main St": (-1, 0), "1st Ave": (1, 0)}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

# panel 1 — primal
ax = axes[0]
ax.set_title("Primal\nnodes = intersections · edges = segments", fontsize=11)
for u, v in P_toy.edges():
    ax.plot([P_pos[u][0], P_pos[v][0]], [P_pos[u][1], P_pos[v][1]],
            color=COL[street_of((u, v))], lw=3, zorder=1)
for n, (x, y) in P_pos.items():
    ax.scatter([x], [y], s=430, color="#333", zorder=2)
    ax.text(x, y, n, color="white", ha="center", va="center", fontsize=10, zorder=3)
ax.text(3.0, 0.35, "Main St", color=COL["Main St"], fontsize=9)
ax.text(0.15, 1.75, "1st Ave", color=COL["1st Ave"], fontsize=9)

# panel 2 — naive line graph
ax = axes[1]
ax.set_title(f"Naive nx.line_graph\n{L_toy.number_of_nodes()} nodes — one per SEGMENT",
             fontsize=11)
for a, b in L_toy.edges():
    ax.plot([L_pos[a][0], L_pos[b][0]], [L_pos[a][1], L_pos[b][1]],
            color="#bbb", lw=1.4, zorder=1)
for n, (x, y) in L_pos.items():
    ax.scatter([x], [y], s=1000, color=COL[street_of(n)], zorder=2)
    ax.text(x, y, f"{n[0]}{n[1]}", color="white", ha="center", va="center",
            fontsize=8, zorder=3)
ax.text(0.0, -2.4, "segments meeting at B interlink into a clique", color="#555",
        ha="center", fontsize=8.5)

# panel 3 — true ICN dual
ax = axes[2]
ax.set_title(f"True ICN dual\n{D_toy.number_of_nodes()} nodes — one per NAMED STREET",
             fontsize=11)
ax.plot([D_pos["Main St"][0], D_pos["1st Ave"][0]], [0, 0], color="#bbb", lw=2, zorder=1)
for n, (x, y) in D_pos.items():
    ax.scatter([x], [y], s=2800, color=COL[n], zorder=2)
    ax.text(x, y, n, color="white", ha="center", va="center", fontsize=9, zorder=3)
ax.text(0.0, -0.9, "share intersection B -> one link", color="#555",
        ha="center", fontsize=8.5)

for ax in axes:
    ax.set_aspect("equal"); ax.axis("off")
fig.suptitle("Same junction, three graphs: the line graph keeps every segment "
             "(and cliques them at each intersection);\nthe ICN dual collapses "
             "each named street to ONE node — that is why we build it ourselves.",
             fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Betweenness on the DUAL (unweighted — links are intersections, not lengths).
# Same move as the primal (Section 4): NetworkX's betweenness is pure-Python and
# O(V*E), and the ICN dual is DENSE (every street sharing an intersection is
# linked), so nx.betweenness_centrality(D) can run for minutes. igraph (C-backed)
# returns the SAME EXACT values in seconds. Convert D -> igraph once, keeping a
# node-index <-> street-name mapping so results map straight back onto streets.
import igraph as ig

dual_ids = list(D.nodes())                          # igraph index i -> street name
didx_of = {s: i for i, s in enumerate(dual_ids)}    # street name    -> igraph index
gd = ig.Graph(
    n=len(dual_ids),
    edges=[(didx_of[a], didx_of[b]) for a, b in D.edges()],
    directed=False,
)

# igraph returns RAW (unnormalized) shortest-path counts; rescale by
# 2/((n-1)(n-2)) — the exact undirected normalization nx.betweenness_centrality
# applies — so values match nx.betweenness_centrality(D, normalized=True).
btw_dual_raw = gd.betweenness(directed=False)
nd = gd.vcount()
dscale = (2.0 / ((nd - 1) * (nd - 2))) if nd > 2 else 1.0
btw_dual = {dual_ids[i]: btw_dual_raw[i] * dscale for i in range(nd)}
print(f"computed dual betweenness (igraph, C-backed) on {nd} street-nodes; "
      f"normalization raw * 2/((n-1)(n-2))")

### The payoff overlay — primal vs. dual betweenness, interactively

**Primal betweenness** (per intersection) and **dual betweenness** (per named street, painted back onto that street's segments) on **one interactive map** with two toggleable layers over the same three basemaps (OpenStreetMap / Esri satellite / blank). Flip between them live.

The primal looks near-regular/planar; the dual reveals a few long, well-connected streets carrying disproportionate through-movement — a different, more scale-free-like story. *"Is this city scale-free?" has no answer until you fix the representation.* That is **no neutral graph**, made concrete.

Same **CRS gotcha** as Section 5: betweenness is computed on the UTM graph; we reproject the geometry to EPSG:4326 only to render it.

> **Instructor fallback (budget safety):** if the live build above ran long, set `USE_FALLBACK = True` to display a pre-rendered overlay PNG instead. That PNG is an optional **slide/fallback asset** generated offline by `make_overlay_asset.py`; set `OVERLAY_PNG_URL` to its raw URL. Leave `USE_FALLBACK = False` for the live interactive result (the default).

In [ ]:
import folium
import branca.colormap as cm

USE_FALLBACK = False  # instructor flips to True only if running over budget
# Optional: raw URL of the pre-rendered overlay PNG (slide/fallback asset,
# generated offline by make_overlay_asset.py). Only needed if USE_FALLBACK.
OVERLAY_PNG_URL = ""  # e.g. ".../unit-1-graph-substrate/data/jerusalem_dual_overlay.png"

if USE_FALLBACK:
    from IPython.display import Image, display
    fb = "jerusalem_dual_overlay.png"
    if not os.path.exists(fb):
        if not OVERLAY_PNG_URL:
            raise SystemExit(
                "USE_FALLBACK is True but OVERLAY_PNG_URL is empty. Set it to "
                "the raw URL of the pre-rendered overlay PNG, or set "
                "USE_FALLBACK = False to run the live build."
            )
        r = requests.get(OVERLAY_PNG_URL, timeout=60); r.raise_for_status()
        open(fb, "wb").write(r.content)
    display(Image(fb))
else:
    # CRS gotcha (again): metrics are on the UTM graph; reproject geometry to
    # EPSG:4326 FOR RENDERING ONLY. Reuse the nodes/edges GDFs idiom.
    o_nodes, o_edges = ox.convert.graph_to_gdfs(G_lcc)
    o_nodes = o_nodes.to_crs(4326)
    o_edges = o_edges.to_crs(4326)

    # Primal betweenness -> per node.
    o_nodes["bc_primal"] = [btw.get(n, 0.0) for n in o_nodes.index]
    # Dual betweenness -> per named street, mapped onto member segments.
    o_edges = o_edges.reset_index()  # expose u, v, key columns
    o_edges["bc_dual"] = [
        btw_dual.get(edge_to_street.get((u, v, k)), 0.0)
        for u, v, k in zip(o_edges["u"], o_edges["v"], o_edges["key"])
    ]

    center = [o_nodes.geometry.y.mean(), o_nodes.geometry.x.mean()]
    m = folium.Map(location=center, zoom_start=14, tiles=None, control_scale=True)
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/"
              "World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles © Esri", name="Esri satellite", overlay=False, control=True,
    ).add_to(m)
    _BLANK = ("data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwC"
              "AAAAC0lEQVR42mNk+A8AAQUBAScY42YAAAAASUVORK5CYII=")
    folium.TileLayer(tiles=_BLANK, attr="blank", name="Blank (topology only)",
                     overlay=False, control=True).add_to(m)

    # Faint full network backdrop so both layers read against the street grid
    # (esp. on the blank basemap).
    folium.GeoJson(
        o_edges[["geometry"]],
        style_function=lambda _f: {"color": "#cccccc", "weight": 1, "opacity": 0.5},
        name="Full network (faint)",
    ).add_to(m)

    # Layer 1: PRIMAL betweenness (top-decile intersections). Was a flat blue
    # dot; now each node is COLOURED *and* SIZED by its betweenness (blue ramp
    # keeps the primal identity), so the busiest junctions stand out from the
    # merely-above-threshold ones instead of all reading as one blue blob.
    pvals = o_nodes["bc_primal"]
    p_cut = float(pvals.quantile(0.90))
    p_top = o_nodes[pvals >= p_cut]
    p_lo = float(p_top["bc_primal"].min())
    p_hi = float(p_top["bc_primal"].max())
    primal_ramp = cm.LinearColormap(
        ["#deebf7", "#9ecae1", "#4292c6", "#08306b"], vmin=p_lo, vmax=p_hi,
        caption="Primal betweenness (top-10% intersections)",
    )
    fg_p = folium.FeatureGroup(name="Primal betweenness (intersections)", show=True)
    for _, row in p_top.iterrows():
        v = float(row["bc_primal"])
        frac = (v - p_lo) / (p_hi - p_lo) if p_hi > p_lo else 1.0
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=3 + 7 * frac, color=primal_ramp(v), fill=True,
            fill_color=primal_ramp(v), fill_opacity=0.9, weight=0,
            tooltip=f"primal BC: {v:.4g}",
        ).add_to(fg_p)
    fg_p.add_to(m)

    # Layer 2: DUAL betweenness (named streets, painted onto member segments).
    # Colour AND line-width both scale with betweenness, so the few arterial
    # streets separate clearly from the faint low-betweenness majority (a single
    # thin ramp made neighbouring streets hard to tell apart).
    dmax = float(o_edges["bc_dual"].max()) or 1.0
    dual_ramp = cm.LinearColormap(
        ["#ffffb2", "#fecc5c", "#fd8d3c", "#e31a1c", "#800026"],
        vmin=0.0, vmax=dmax, caption="Dual betweenness (named streets)",
    )
    def _dual_width(v):
        return 2 + 6 * (v / dmax)
    def _dual_style(f):
        v = f["properties"]["bc_dual"]
        return {"color": dual_ramp(v), "weight": _dual_width(v), "opacity": 0.95}
    # Leaflet polylines have no outline, so "case" each street: a slightly wider
    # BLACK line underneath, the coloured line on top. The thin black border on
    # both sides makes adjacent streets easy to tell apart where they run close.
    def _dual_casing(f):
        v = f["properties"]["bc_dual"]
        return {"color": "#000000", "weight": _dual_width(v) + 2.2, "opacity": 0.9}
    fg_d = folium.FeatureGroup(name="Dual betweenness (named streets)", show=False)
    folium.GeoJson(
        o_edges[["geometry", "bc_dual"]],
        style_function=_dual_casing, name="dual-casing", control=False,
    ).add_to(fg_d)                                    # black border, drawn first
    folium.GeoJson(
        o_edges[["geometry", "bc_dual"]],
        style_function=_dual_style, name="dual", control=False,
    ).add_to(fg_d)                                    # coloured line, on top
    fg_d.add_to(m)

    # Colour legends for both metrics (always visible, act as the map key).
    primal_ramp.add_to(m)
    dual_ramp.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    display(m)

## 7. Which graph + metric = arteries? — and the honest caveat

**Concept (vocabulary):** *match graph + metric to the question; and know the limit — betweenness ≠ realized flow.*

**Matching table — pick the tool for the question:**

| Planner's question | Graph + metric |
|---|---|
| Which streets carry through-traffic? | **betweenness on the primal** |
| Which places are most accessible? | **closeness** |
| Which streets structure the city as named corridors? | **dual** (named-street) |
| Which streets are most damaging to remove? | information centrality |

**The honest caveat (planted, not belabored).** A high-betweenness map is a *structural prediction* about through-movement — **not measured flow.** Real flow also depends on demand, origins/destinations, and time of day, which pure topology omits (Kazerani & Winter 2009; Gao et al. 2013). That gap is exactly where the practice extension and Units 3–4 begin. We state it once, clearly, and move on.

## 8. Going deeper (OPTIONAL — *not* part of the 45-min core)

> **Time budget note.** Everything below is an **optional appendix**. It is
> deliberately **outside** the 45-minute demo budget — skip it live and point
> students to it as take-home. It is self-contained and runs on a **small
> sub-area** (an ego-subgraph around one intersection) so each cell finishes in
> **seconds**, not minutes.

The meta-lesson of this unit is *there is no neutral graph*. So far that was a
statement about which streets *look* important. Here it becomes **algorithmic**:
the representation decides which questions are even **expressible**. Two
demonstrations:

- **O1 — turn restrictions / penalties.** A primal shortest path is *turn-blind*:
  it can route straight through a banned left turn. A **turn-aware expanded
  (line-graph) dual** — nodes = directed road segments, links = *legal*
  movements carrying a turn penalty — can express "no left turn here" and
  routes around it. We show the two routes differ.
- **O2 — fewest-turns path.** On the named-street dual from Section 6, shortest
  path by **hop count** = the route that changes streets the fewest times. We
  compare it against the shortest-distance route.

### O1 — Turn restrictions: build a turn-aware expanded dual

A primal graph cannot say "you may not turn left from A onto B": a turn is a
*pair of consecutive edges*, and the primal only knows single edges. The fix is
a standard **edge-based expansion** (a turn-aware line-graph dual):

- **Nodes** = directed primal segments `(u, v)`.
- **Links** = legal movements `(u, v) → (v, w)` at the shared node `v`, each
  weighted by `length(v, w) + turn_penalty(angle at v)`.
- A **banned turn** is simply a movement we **omit** from the dual.

Routing on this dual respects turn legality and turn cost; routing on the primal
does not. We build it on a **small sub-area** so it runs in seconds, plant one
banned left turn on the primal shortest path, and compare.

**The picture.** A turn restriction is impossible to express in the primal but
trivial in the dual. The schematic below makes the difference concrete: in the
**primal** a route through intersection `v` has forgotten which edge it arrived
on, so "no left turn from `u`" is unsayable; in the **dual / line graph** every
*movement* is an edge, so you forbid a turn by **deleting a single edge**.

In [ ]:
# Schematic (we draw it): why the DUAL can model a turn ban and the PRIMAL can't.
import matplotlib.pyplot as plt

fig, (axp, axd) = plt.subplots(1, 2, figsize=(13, 5.6))

# ---------- LEFT: primal intersection ----------
axp.set_title("Primal graph: nodes = intersections\n"
              "a turn is a PAIR of edges — the graph can't forbid it", fontsize=11)
P = {"u": (0, -1), "v": (0, 0), "L": (-1, 0), "R": (1, 0), "S": (0, 1)}
for a, b in [("u", "v"), ("v", "L"), ("v", "R"), ("v", "S")]:
    axp.plot([P[a][0], P[b][0]], [P[a][1], P[b][1]], color="#999", lw=2, zorder=1)
for k, (x, y) in P.items():
    axp.scatter([x], [y], s=320, color="#333", zorder=2)
    if k in ("u", "v"):
        axp.text(x, y, k, color="white", ha="center", va="center", fontsize=11, zorder=3)
# the movement u -> v -> L (a left turn), drawn as two arrows
axp.annotate("", xy=(0, -0.18), xytext=(0, -0.82),
             arrowprops=dict(arrowstyle="-|>", color="#1f5fae", lw=2.6))
axp.annotate("", xy=(-0.82, 0), xytext=(-0.18, 0),
             arrowprops=dict(arrowstyle="-|>", color="#d62728", lw=2.6))
axp.scatter([-0.5], [0.0], s=650, marker="x", color="red", linewidths=4, zorder=4)
axp.text(-0.5, 0.22, "no left turn", color="red", ha="center", fontsize=9.5)
axp.text(0.0, -1.28, "arriving from u…", ha="center", fontsize=9, color="#1f5fae")
axp.text(0.78, -0.5, "a path through v\nforgets it came from u",
         fontsize=8.5, color="#555", ha="center")
axp.set_xlim(-1.7, 1.9); axp.set_ylim(-1.6, 1.4); axp.axis("off"); axp.set_aspect("equal")

# ---------- RIGHT: dual / line graph ----------
axd.set_title("Dual / line graph: nodes = directed segments\n"
              "a turn IS an edge — delete it to forbid the turn", fontsize=11)
D = {"uv": (0, -1), "vS": (0, 1), "vL": (-1.2, 0.2), "vR": (1.2, 0.2)}
labels = {"uv": "(u,v)", "vS": "(v,S)", "vL": "(v,L)", "vR": "(v,R)"}
for a, b, ok in [("uv", "vS", True), ("uv", "vR", True), ("uv", "vL", False)]:
    col = "#2ca02c" if ok else "red"
    lst = "solid" if ok else (0, (4, 3))
    axd.annotate("", xy=D[b], xytext=D[a],
                 arrowprops=dict(arrowstyle="-|>", color=col, lw=2.6,
                                 linestyle=lst, shrinkA=20, shrinkB=20))
for k, (x, y) in D.items():
    axd.scatter([x], [y], s=1100, color="#444", zorder=2)
    axd.text(x, y, labels[k], color="white", ha="center", va="center",
             fontsize=9.5, zorder=3)
mid = ((D["uv"][0] + D["vL"][0]) / 2, (D["uv"][1] + D["vL"][1]) / 2)
axd.scatter([mid[0]], [mid[1]], s=750, marker="x", color="red", linewidths=4, zorder=4)
axd.text(mid[0] - 0.1, mid[1] - 0.3, "delete this edge\n= ban the turn",
         color="red", ha="center", fontsize=8.5)
axd.text(0.0, 1.45, "movements = edges (legal in green)", ha="center",
         fontsize=9, color="#2ca02c")
axd.set_xlim(-2.0, 2.0); axd.set_ylim(-1.7, 1.8); axd.axis("off"); axd.set_aspect("equal")

fig.suptitle("Restriction modelling: the primal can't express a turn ban; "
             "the dual just deletes one edge", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
import math, time, folium
import networkx as nx
import pandas as pd

# --- Sub-area: a bigger ego-ball so the demonstrated route is genuinely LONG.
SUB_CAP = 1200   # nodes; the turn-expansion (~deg^2 per node) still runs in seconds
hub = max(G_lcc.degree, key=lambda kv: kv[1])[0]
sub_nodes = {hub}
for _hop in range(12):
    frontier = {nbr for n in list(sub_nodes) for nbr in G_lcc.neighbors(n)}
    if len(sub_nodes | frontier) > SUB_CAP:
        for nbr in frontier:                  # top up toward the cap
            if len(sub_nodes) >= SUB_CAP:
                break
            sub_nodes.add(nbr)
        break
    sub_nodes |= frontier
H = G_lcc.subgraph(sub_nodes).copy()
H = H.subgraph(max(nx.connected_components(H.to_undirected()), key=len)).copy()
print(f"sub-area: {H.number_of_nodes()} nodes, {H.number_of_edges()} edges")

# Geographic coords (UTM, meters) for bearing / turn-angle math.
xy = {n: (H.nodes[n]["x"], H.nodes[n]["y"]) for n in H.nodes}

def bearing(a, b):
    (ax, ay), (bx, by) = xy[a], xy[b]
    return math.degrees(math.atan2(bx - ax, by - ay))   # 0=N, clockwise

def turn_angle(u, v, w):
    """Signed turn at v going u->v->w in [-180,180]; +left, -right (approx)."""
    return (bearing(v, w) - bearing(u, v) + 180) % 360 - 180

def seg_len(a, b):
    if GG.has_edge(a, b):
        return float(GG[a][b]["length"])
    return math.dist(xy[a], xy[b])

# Simple undirected metric graph for the (turn-blind) primal route.
GH = nx.Graph()
for a, b in H.to_undirected().edges():
    GH.add_edge(a, b, length=seg_len(a, b))

# --- Pick a LONG origin -> destination (double sweep ~ graph diameter). ------
def farthest_from(src):
    d = nx.shortest_path_length(GH, src, weight="length")
    return max(d, key=d.get)
orig = farthest_from(next(iter(GH.nodes)))
dest = farthest_from(orig)

# --- (i) PRIMAL shortest path: turn-blind. -----------------------------------
t0 = time.perf_counter()
primal_path = nx.shortest_path(GH, orig, dest, weight="length")
primal_t = time.perf_counter() - t0
primal_len = sum(seg_len(primal_path[i], primal_path[i + 1])
                 for i in range(len(primal_path) - 1))
print(f"long primal route: {len(primal_path)} nodes, {primal_len:.0f} m")

# --- (ii) Build the turn-aware DUAL ONCE: nodes=segments, links=legal moves. -
TURN_PENALTY = {"left": 60.0, "right": 8.0, "straight": 0.0, "uturn": 1e6}
def turn_class(ang):
    if abs(ang) < 25:  return "straight"
    if abs(ang) > 150: return "uturn"
    return "left" if ang > 0 else "right"

t0 = time.perf_counter()
Dturn = nx.DiGraph()
# Build over the SAME undirected adjacency the primal route uses (GH), so
# every street segment exists in BOTH directions. Using H.neighbors() here
# returns successors only on the directed MultiDiGraph subgraph, so the
# route's first/last segment could be absent -> NetworkXNoPath -> empty route.
for u in GH.nodes:
    for v in GH.neighbors(u):                # incoming segment (u, v)
        for w in GH.neighbors(v):            # outgoing segment (v, w)
            if w == u:                       # skip immediate U-turn back
                continue
            pen = TURN_PENALTY[turn_class(turn_angle(u, v, w))]
            Dturn.add_edge((u, v), (v, w), weight=seg_len(v, w) + pen)
dual_build_t = time.perf_counter() - t0

s_start, s_end = (primal_path[0], primal_path[1]), (primal_path[-2], primal_path[-1])

def dual_route_nodes(omit=None):
    """Turn-aware route s_start->s_end, optionally omitting ONE movement edge."""
    saved = None
    if omit is not None and Dturn.has_edge(*omit):
        saved = (omit, Dturn[omit[0]][omit[1]]["weight"])
        Dturn.remove_edge(*omit)
    try:
        segs = nx.shortest_path(Dturn, s_start, s_end, weight="weight")
        nodes = [segs[0][0]] + [s[1] for s in segs]
    except nx.NetworkXNoPath:
        nodes = []
    if saved is not None:
        Dturn.add_edge(saved[0][0], saved[0][1], weight=saved[1])
    return nodes

# --- Choose the restriction that MOST changes the route (so it's obvious). ---
# Candidates = interior turns ON the primal route; ban the most impactful one.
cands = [((primal_path[i - 1], primal_path[i]), (primal_path[i], primal_path[i + 1]),
          (primal_path[i - 1], primal_path[i], primal_path[i + 1]))
         for i in range(1, len(primal_path) - 1)]
if len(cands) > 25:                          # cap the search for runtime
    cands = cands[::len(cands) // 25 + 1]

t0 = time.perf_counter()
banned, dual_path, best_div = None, dual_route_nodes(), -1
for e_from, e_to, mv in cands:
    nodes = dual_route_nodes(omit=(e_from, e_to))
    if not nodes:
        continue
    div = len(set(primal_path) ^ set(nodes))  # symmetric node difference
    if div > best_div:
        best_div, banned, dual_path = div, mv, nodes
search_t = time.perf_counter() - t0
dual_t = dual_build_t + search_t

dual_len = (sum(seg_len(dual_path[i], dual_path[i + 1])
                for i in range(len(dual_path) - 1)) if dual_path else 0)
print(f"most-impactful banned turn (u->v->w): {banned}")
print(f"turn-aware dual route: {len(dual_path)} nodes, {dual_len:.0f} m; "
      f"divergence from primal (differing nodes) = {best_div}")

traverses_ban = banned is not None and any(
    (primal_path[i - 1], primal_path[i], primal_path[i + 1]) == banned
    for i in range(1, len(primal_path) - 1))

print(pd.DataFrame({
    "metric": ["nodes in graph", "route length (m)", "build+route runtime (s)",
               "route uses the banned (illegal) turn?"],
    "primal (turn-blind)": [GH.number_of_nodes(), round(primal_len), round(primal_t, 4),
                            "YES - unrealistic" if traverses_ban else "no"],
    "turn-aware dual":     [Dturn.number_of_nodes(), round(dual_len), round(dual_t, 3),
                            "no - legal + penalized"],
}).to_string(index=False))


**Render — toggle each route on/off.** The primal (turn-blind) route is blue, the turn-aware dual route is red, and **each is its own layer** in the control (top-right) so you can flip them on/off and compare. The yellow dot is the banned turn — chosen as the *most impactful* restriction on the primal route, so the legal dual route is forced to visibly detour around it. Basemaps (OpenStreetMap / Esri satellite / blank) are switchable too.

In [ ]:
# Render: each route is its OWN toggleable layer (play with on/off), over
# switchable basemaps. CRS gotcha: reproject sub-area node coords to WGS84.
sub_gdf = ox.convert.graph_to_gdfs(H, edges=False).to_crs(4326)
sub_edges = ox.convert.graph_to_gdfs(H, nodes=False).to_crs(4326)
ll = {n: (sub_gdf.geometry.y.loc[n], sub_gdf.geometry.x.loc[n]) for n in H.nodes}
center = [sum(ll[n][0] for n in H.nodes) / H.number_of_nodes(),
          sum(ll[n][1] for n in H.nodes) / H.number_of_nodes()]

_ESRI = ("https://server.arcgisonline.com/ArcGIS/rest/services/"
         "World_Imagery/MapServer/tile/{z}/{y}/{x}")
_BLANK = ("data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwC"
          "AAAAC0lEQVR42mNk+A8AAQUBAScY42YAAAAASUVORK5CYII=")

m = folium.Map(location=center, zoom_start=15, tiles=None, control_scale=True)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(tiles=_ESRI, attr="Tiles © Esri", name="Esri satellite",
                 overlay=False, control=True).add_to(m)
folium.TileLayer(tiles=_BLANK, attr="blank", name="Blank (topology only)",
                 overlay=False, control=True).add_to(m)

# Faint sub-area network for context.
folium.GeoJson(
    sub_edges[["geometry"]],
    style_function=lambda _f: {"color": "#cccccc", "weight": 1, "opacity": 0.5},
    name="Sub-area network (faint)",
).add_to(m)

def route_layer(path, color, name, show=True):
    fg = folium.FeatureGroup(name=name, show=show)   # its own on/off layer
    if path:
        folium.PolyLine([ll[n] for n in path], color=color, weight=6,
                        opacity=0.85, tooltip=name).add_to(fg)
    fg.add_to(m)

route_layer(primal_path, "#1f5fae", "Primal route (turn-blind - may be illegal)")
route_layer(dual_path,   "#d62728", "Turn-aware dual route (legal + penalized)")

# A / B / banned-turn markers as their own toggleable layer.
fg_mk = folium.FeatureGroup(name="A / B / banned turn", show=True)
folium.Marker(ll[orig], tooltip="A (origin)",
              icon=folium.Icon(color="green")).add_to(fg_mk)
folium.Marker(ll[dest], tooltip="B (destination)",
              icon=folium.Icon(color="blue")).add_to(fg_mk)
if banned is not None:
    folium.CircleMarker(ll[banned[1]], radius=8, color="black", fill=True,
                        fill_color="yellow", fill_opacity=1.0,
                        tooltip=f"banned turn here: {banned}").add_to(fg_mk)
fg_mk.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m


### O2 — Fewest-turns path (a lighter second look)

Reuse the **named-street dual** from Section 6. There, hop count between two
street-nodes = **number of street changes**. So a shortest path *by hop count*
on that dual is the route with the **fewest turns** (street changes) — often the
one a human can actually remember ("take King George all the way, then Jaffa").
We compare it against the shortest-*distance* route between the same two points.

In [ ]:
# Map the O1 origin/destination to their named streets in the Section-6 dual.
def street_of(node):
    """A street-node incident to `node` in the primal (first match)."""
    for u, v, k in G_lcc.edges(node, keys=True):
        s = edge_to_street.get((u, v, k)) or edge_to_street.get((v, u, k))
        if s is not None:
            return s
    return None

s_orig, s_dest = street_of(orig), street_of(dest)
print("origin street:", s_orig, "| destination street:", s_dest)

if s_orig in D and s_dest in D and nx.has_path(D, s_orig, s_dest):
    # Fewest-turns = unweighted shortest path (hops) on the named-street dual.
    fewest_turns = nx.shortest_path(D, s_orig, s_dest)   # sequence of streets
    n_changes = len(fewest_turns) - 1
    # Min-distance route (primal, meters) for contrast = the O1 primal_path.
    min_dist_m = sum(seg_len(primal_path[i], primal_path[i+1])
                     for i in range(len(primal_path) - 1))
    print(f"fewest-turns route: {n_changes} street changes")
    print("  streets:", " -> ".join(map(str, fewest_turns)))
    print(f"min-distance route: {len(primal_path)-1} segments, "
          f"{min_dist_m:.0f} m")
    print("\nReading: 'fewest turns' wins for human-followability when the "
          "shortest-distance route zig-zags across many short streets; "
          "'shortest distance' wins for a vehicle minimizing travel time.")
else:
    print("origin/destination streets not connected in the dual sub-sample; "
          "pick a different A/B (this is an optional take-home cell).")

### Why this matters

The **representation determines which questions are even expressible**. Turn
legality lives in the *expanded dual* and is invisible to the primal;
"fewest street-changes" lives in the *named-street dual* and is meaningless on
the primal. *No neutral graph* — now algorithmic, not just visual.

## Where to go next

Open-ended explorations — pick one and run the **direct → interpret → extend** loop:

1. **Compute a centrality we skipped** (straightness or information) and redraw the top-10% map. What *question* does it answer that betweenness doesn't — and does its "important" set overlap betweenness's?
2. **Pick your own city** via the live-pull showcase (`RUN_LIVE = True` on a small place), and compare its centrality / meshedness fingerprint against Jerusalem. Is your city's betweenness more or less concentrated? (You can sanity-check against Boeing's [Harvard Dataverse](https://doi.org/10.7910/DVN/KA5HJ3) per-city indicators — is your number high or low globally?)
3. **Remove a street as car-routable** (a preview of the light-rail practice task) and recompute betweenness. Which streets become *more* critical — and what does "more critical" mean structurally vs. in realized traffic?
